In [1]:
import pickle
import numpy as np
from astropy.cosmology import Flatw0waCDM
from matplotlib import pyplot as plt
from pdspl_utils.inference import draw_lens_from_given_zs, run_dspl_inference
import corner
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

In [2]:
POSTERIOR_DIRECTORY = './posteriors_uniform_split_selection'

## Prepare Datasets for the Forecast

In [3]:
# Load the pdspl_samples object
with open('../data/samples/pdspl_samples_with_pairs.pkl', 'rb') as f:
    pdspl_samples = pickle.load(f)

In [4]:
# --- 1. CONFIGURATION & TRUTH ---
truth = {
    'h0': 70.0, 'om': 0.3, 'w0': -1.0, 'wa': 0.0,
    'lambda_int': 1.0, 'lambda_sigma': 0.05,
    'gamma_pl': 2.0, 'gamma_sigma': 0.16
}
fixed_params = {'h0': 70.0}
cosmo_true = Flatw0waCDM(H0=truth['h0'], Om0=truth['om'], w0=truth['w0'], wa=truth['wa'])

In [ ]:
# --- 2. INITIALIZE SCENARIOS ---
dissimilarity_cuts = np.linspace(0.01, 0.1, 19)
colors = plt.cm.viridis(np.linspace(0, 1, len(dissimilarity_cuts)))

scenarios = {}

org_table = pdspl_samples['lsst_y10']['table']
org_pairs_table = pdspl_samples['lsst_y10']['pairs_analysis']['pairs_table']
org_pairs_table_with_errors = pdspl_samples['lsst_y10']['pairs_analysis']['pairs_table_with_errors']

for i, cut in enumerate(dissimilarity_cuts):
    mask = org_pairs_table_with_errors['dissimilarity'] <= cut
    
    pairs_table_cut = org_pairs_table[mask]
    pairs_table_with_errors_cut = org_pairs_table_with_errors[mask]

    num_pairs_cut = len(pairs_table_cut)
    down_sampling = num_pairs_cut / 500 if num_pairs_cut > 500 else 1

    sigma_beta = np.std(1 - pairs_table_cut['beta_E_pseudo']/pairs_table_cut['beta_E_DSPL'])

    key = f'lsst_y10_dissim_leq_{cut:.3f}'
    name = r"PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $"+ f"{cut:.3f})"
    if cut == np.inf:
        name = "LSST Y10 (Full Sample)"
        key = 'lsst_y10_full_sample'


    scenarios[key] = {
        'name': name,
        'down_sampling': down_sampling,
        'num_lenses': num_pairs_cut,
        'color': colors[i],
        'coeffs': pdspl_samples['lsst_y10']['pairs_analysis']['scatter_vs_dissimilarity_fit_coeffs'],
        'pairs_table': pairs_table_cut,
        'pairs_table_with_errors': pairs_table_with_errors_cut,
        'sigma_beta': sigma_beta,
        'redshift_error_rel': 0.03,
        'backend_path': POSTERIOR_DIRECTORY + f"/lsst_y10_dissim_leq_{cut:.3f}.h5"
    }

# --- 3. GENERATE MOCK DATASETS ---
print(f"{'SCENARIO':<60} | {'N_LENS':<10} | {'STATUS'}")
print("-" * 85)

for key in scenarios.keys():
    sc = scenarios[key]
    num_samples = int(sc['num_lenses'] / sc['down_sampling'])
    kwargs_list = []
    
    # Generation from PDSPL tables
    pairs_table = sc['pairs_table']
    pairs_table_with_errors = sc['pairs_table_with_errors']
    coeffs = sc['coeffs']
    random_indices = np.random.choice(len(pairs_table), size=num_samples, replace=False)
    
    for i in random_indices:
        z_lens = pairs_table["z_D"][i]
        z1 = pairs_table["z_S1"][i]
        z2 = pairs_table["z_S2"][i]
        
        dissimilarity = pairs_table_with_errors['dissimilarity'][i]
        rel_scatter_in_beta_E = np.polyval(coeffs, dissimilarity)
        
        sigma_beta_meas, sigma_beta_los = 0.01, 0.01
        sigma_meas_rel = np.sqrt(sigma_beta_meas**2 + sigma_beta_los**2)
        
        kwargs_list.append(draw_lens_from_given_zs(
            z_lens=z_lens, z1=z1, z2=z2,
            lambda_mst_mean=truth['lambda_int'], lambda_mst_sigma=truth['lambda_sigma'],
            gamma_pl_mean=truth['gamma_pl'], gamma_pl_sigma=truth['gamma_sigma'],
            sigma_meas_rel = sigma_meas_rel,
            sigma_beta_intrinsic= rel_scatter_in_beta_E, 
            down_sampling=sc['down_sampling'], 
            with_noise=False, # For ASIMOV inference
            cosmo=cosmo_true, 
            redshift_error_rel=sc['redshift_error_rel']
        ))
            
    # Save the generated dataset into the scenario dictionary
    sc['kwargs_list'] = kwargs_list
    print(f"{sc['name']:<60} | {num_samples:<10} | GENERATED")

SCENARIO                                                     | N_LENS     | STATUS
-------------------------------------------------------------------------------------
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.010)   | 500        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.015)   | 500        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.020)   | 500        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.025)   | 499        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.030)   | 500        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.035)   | 500        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.040)   | 500        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.045)   | 500        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.050)   | 500        | GENERATED
PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.055

## Run the Forecast Inference

### a. Uniform Prior Forecasts

In [8]:
scenarios.keys()

dict_keys(['lsst_y10_dissim_leq_0.010', 'lsst_y10_dissim_leq_0.015', 'lsst_y10_dissim_leq_0.020', 'lsst_y10_dissim_leq_0.025', 'lsst_y10_dissim_leq_0.030', 'lsst_y10_dissim_leq_0.035', 'lsst_y10_dissim_leq_0.040', 'lsst_y10_dissim_leq_0.045', 'lsst_y10_dissim_leq_0.050', 'lsst_y10_dissim_leq_0.055', 'lsst_y10_dissim_leq_0.060', 'lsst_y10_dissim_leq_0.065', 'lsst_y10_dissim_leq_0.070', 'lsst_y10_dissim_leq_0.075', 'lsst_y10_dissim_leq_0.080', 'lsst_y10_dissim_leq_0.085', 'lsst_y10_dissim_leq_0.090', 'lsst_y10_dissim_leq_0.095', 'lsst_y10_dissim_leq_0.100'])

In [6]:
# --- 1. INFERENCE SETTINGS ---
my_guess = {
    'om': 0.3, 'w0': -1.0, 'wa': 0.0,
    'lambda_int': 1.0, 'lambda_sigma': 0.05,
    'gamma_pl': 2.0, 'gamma_sigma': 0.16
}

my_spreads = {
    'om': 0.1, 'w0': 0.1, 'wa': 0.1,
    'lambda_int': 0.1, 'lambda_sigma': 0.01,
    'gamma_pl': 0.1, 'gamma_sigma': 0.01
}

my_priors = {
    'h0': ('uniform', 50.0, 90.0),
    'om': ('uniform', 0.0, 1.0),
    'w0': ('uniform', -3.0, 0.0),
    'wa': ('uniform', -5.0, 5.0),
    'lambda_int': ('uniform', 0.8, 1.2),
    'lambda_sigma': ('uniform', 0.0, 0.2),
    'gamma_pl': ('uniform', 1.0, 3.0),
    'gamma_sigma': ('uniform', 0.0, 0.5)
}

# --- 2. EXECUTION LOOP ---
print("\n--- Starting MCMC Samplers ---")

param_labels = None 

for key in scenarios.keys():
    sc = scenarios[key]
    
    if 'kwargs_list' not in sc:
        continue # Skip if data wasn't generated
        
    print(f"\nRunning Inference for: {sc['name']}")
    
    samples, labels = run_dspl_inference(
        sc['kwargs_list'],
        n_walkers=64,
        n_steps=1000, 
        n_burn=500,
        initial_guess=my_guess,
        fixed_params=fixed_params,
        initial_scatter=my_spreads,
        priors=my_priors,
        down_sampling=sc['down_sampling'],
        backend_path=sc['backend_path']
    )
    
    sc['samples'] = samples
    param_labels = labels # Saves labels for plotting step
    print(f"{sc['name']} | INFERENCE COMPLETE")


--- Starting MCMC Samplers ---

Running Inference for: PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.010)
--- Starting DSPL Inference ---
Fixed Params  : ['h0']
Sampled Params: ['om', 'w0', 'wa', 'lambda_int', 'lambda_sigma', 'gamma_pl', 'gamma_sigma']
Down Sampling : 4.622
Saving to     : ./posteriors_uniform_split_selection/lsst_y10_dissim_leq_0.010.h5


100%|██████████| 1000/1000 [03:58<00:00,  4.20it/s]


PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.010) | INFERENCE COMPLETE

Running Inference for: PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.015)
--- Starting DSPL Inference ---
Fixed Params  : ['h0']
Sampled Params: ['om', 'w0', 'wa', 'lambda_int', 'lambda_sigma', 'gamma_pl', 'gamma_sigma']
Down Sampling : 19.642
Saving to     : ./posteriors_uniform_split_selection/lsst_y10_dissim_leq_0.015.h5


100%|██████████| 1000/1000 [04:10<00:00,  3.99it/s]


PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.015) | INFERENCE COMPLETE

Running Inference for: PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.020)
--- Starting DSPL Inference ---
Fixed Params  : ['h0']
Sampled Params: ['om', 'w0', 'wa', 'lambda_int', 'lambda_sigma', 'gamma_pl', 'gamma_sigma']
Down Sampling : 43.454
Saving to     : ./posteriors_uniform_split_selection/lsst_y10_dissim_leq_0.020.h5


...

PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.065) | INFERENCE COMPLETE

Running Inference for: PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.070)
--- Starting DSPL Inference ---
Fixed Params  : ['h0']
Sampled Params: ['om', 'w0', 'wa', 'lambda_int', 'lambda_sigma', 'gamma_pl', 'gamma_sigma']
Down Sampling : 146.248
Saving to     : ./posteriors_uniform_split_selection/lsst_y10_dissim_leq_0.070.h5


100%|██████████| 1000/1000 [04:36<00:00,  3.62it/s]


PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.070) | INFERENCE COMPLETE

Running Inference for: PDSPL (LSST Y10, $\mathcal{D}_{\rm deflector} \leq $0.075)
--- Starting DSPL Inference ---
Fixed Params  : ['h0']
Sampled Params: ['om', 'w0', 'wa', 'lambda_int', 'lambda_sigma', 'gamma_pl', 'gamma_sigma']
Down Sampling : 148.7
Saving to     : ./posteriors_uniform_split_selection/lsst_y10_dissim_leq_0.075.h5


 37%|███▋      | 367/1000 [01:42<02:56,  3.58it/s]


BlockingIOError: [Errno 11] Unable to synchronously open file (unable to lock file, errno = 11, error message = 'Resource temporarily unavailable')